# Домашняя работа 1. Рекомендация товаров (top-10)

**ФИО:** вписать своё ФИО.

**Датасет:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) (541 909 строк в исходном файле). В нём находятся покупки интернет-магазина за 2010–2011 гг.

**Задача:** рекомендовать пользователю 10 товаров, которых он ещё не покупал. Считаем рекомендацию правильной, если пользователь впервые купил товар в будущем периоде.

**Запуск:** установить зависимости через `pip install -r requirements.txt` и выполнить ячейки по порядку. Файл `Online Retail.xlsx` загрузится с UCI автоматически. Если загрузка не сработает, скачать его со страницы датасета в `data/Online Retail.xlsx`. Данные в GitHub не загружаем.

## 1. Загрузка данных

`CustomerID` — покупатель, `StockCode` — товар, `InvoiceNo` — заказ, `InvoiceDate` — дата, `Quantity` — количество, `UnitPrice` — цена, `Description` — название, `Country` — страна покупателя.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from preprocessing import load_original, prepare, interactions_matrix
from evaluation import create_fold, evaluate
from models.item_knn import ItemKNN
from models.ials import ImplicitALS
from models.ease import EASE
from models.fm import FactorizationMachine, MetadataEncoder, IDOnlyEncoder
from models.custom_ials import CustomTargetIALS

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

In [ ]:
raw, file_hash = load_original("data/Online Retail.xlsx")
print("Исходный размер:", raw.shape)
display(raw.head())

## 2. Проверка и подготовка данных

Удаляем дубликаты, пропуски, отменённые заказы, строки с неположительными ценой или количеством и служебные коды товаров. Для ускорения EASE берём **1150 самых популярных товаров по train**, а также покупателей с минимум 3 разными товарами в train. Отбор не использует будущие покупки.

In [ ]:
print("Пропуски:")
display(raw.isna().sum().to_frame("Количество"))
print("Дубликаты:", raw.duplicated().sum())
print("Отмены:", raw["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Пользователей:", raw["CustomerID"].nunique())
print("Товаров:", raw["StockCode"].nunique())

In [ ]:
data = prepare(raw, file_hash, max_items=1150, min_train_items_per_user=3)
n_users = len(data.user_lookup)
n_items = len(data.item_lookup)

sizes = pd.DataFrame({
    "Часть": ["Исходные", "После очистки и выбора товаров", "Итог", "Train", "Validation", "Test"],
    "Строк": [len(raw), len(data.cleaned), len(data.events),
              len(data.train), len(data.validation), len(data.test)]
})
display(sizes)

print("Пользователей:", n_users)
print("Товаров:", n_items)
print("Разреженность:", round(1 - data.events[["u", "i"]].drop_duplicates().shape[0]
                              / (n_users * n_items), 4))

In [ ]:
user_activity = data.events.groupby("CustomerID")["StockCode"].nunique()
item_popularity = data.events.groupby("StockCode")["CustomerID"].nunique()

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].hist(user_activity, bins=30)
axes[0].set_title("Товаров на покупателя")
axes[1].hist(item_popularity, bins=30)
axes[1].set_title("Покупателей на товар")
axes[2].hist(data.train["UnitPrice"].clip(upper=30), bins=30)
axes[2].set_title("Цены в train (срез до 30)")
plt.tight_layout()
plt.show()

print("Стран:", data.train["Country"].nunique())
print("Уникальных названий:", data.train["Description"].nunique())
print("Самые частые страны:")
display(data.train["Country"].value_counts().head(5))
print("Примеры названий:")
display(data.train[["StockCode", "Description", "UnitPrice"]].drop_duplicates("StockCode").head())

## 3. Разделение и метрики

Используем **временное разделение**: первые 70% по времени — train, следующие 15% — validation, последние 15% — test (границы времени рассчитаны после первичной очистки). Все пять моделей используют одни и те же части данных.

На validation обучаемся только на train; на test переобучаемся на train + validation. В рекомендациях убираем все уже купленные товары. Кандидаты — **все товары из train**, а не случайная выборка негативов. Новых покупателей и товары вне train не оцениваем (warm-start). Пользователей без новых покупок в соответствующем периоде не включаем в расчёт метрик.

**Главная метрика до подбора параметров — Recall@10** (доля найденных новых покупок). Вторая — **NDCG@10** (учитывает порядок товаров в топ-10). Выбираем параметры по Recall@10, при равенстве — по NDCG@10.

In [ ]:
print("Конец train:", data.train_cut)
print("Конец validation:", data.val_cut)

train_val = pd.concat([data.train, data.validation], ignore_index=True)
counts_train, qty_train, dates_train = interactions_matrix(data.train, n_users, n_items)
counts_train_val, qty_train_val, dates_train_val = interactions_matrix(train_val, n_users, n_items)

valid = create_fold(data.train, data.validation, n_users, n_items, "validation")
test = create_fold(train_val, data.test, n_users, n_items, "test")

print("Пользователей в validation:", len(valid.users))
print("Пользователей в test:", len(test.users))
assert len(valid.users) > 0 and len(test.users) > 0

## 4. Модели

Все модели реализованы самостоятельно в папке `models/`.

1. **ItemKNN.** Похожие товары ищем по косинусному сходству: $s_{ij} = \frac{X_i^T X_j}{\|X_i\|\|X_j\|+\beta}$. Оценка товара — сумма сходств с покупками пользователя.
2. **iALS.** Для купленных товаров $p_{ui}=1$, для остальных $p_{ui}=0$. Минимизируем $\sum_{ui}c_{ui}(p_{ui}-x_u^Ty_i)^2+\lambda(\|X\|^2+\|Y\|^2)$, где $c_{ui}=1+\alpha\ln(1+n_{ui})$, $n_{ui}$ — число заказов.
3. **EASE.** $G=X^TX+\lambda I$, $P=G^{-1}$, $B_{ij}=-P_{ij}/P_{jj}$ при $i\neq j$, диагональ равна нулю. Предсказание: $XB$.
4. **FM.** $\hat y=w_0+\sum_j w_jx_j+\sum_{j<k}\langle v_j,v_k\rangle x_jx_k$. Признаки: ID покупателя, ID товара, страна, группа цены и два слова из названия. Слова помогают учитывать свойства товаров, а страна и цена — предпочтения покупателя. Признаки берём **только из train**. Обучаем на покупках и случайных неизвестных парах, используя BCE и PyTorch.
5. **iALS с новым таргетом.** Вместо единицы для каждой покупки используем:

$$
F=\min\left(1,\frac{\ln(1+n)}{\ln(1+P_{95}(n))}\right),\qquad
V=\min\left(1,\frac{\ln(1+q)}{\ln(1+P_{95}(q))}\right)
$$
$$
p^*=(0.15+0.85(0.65F+0.35V))\,\left(0.5+0.5\cdot2^{-d/H}\right).
$$

Здесь $q$ — число купленных штук, $d$ — дней с последней покупки до конца обучающего периода, $H$ — период затухания. Для непокупок $p^*=0$. Повторные и недавние покупки получают больший таргет. **Confidence и остальные настройки оставляем как у обычного iALS**: меняется именно целевое значение, а не веса.

## 5. Подбор параметров на validation

Пробуем по два варианта на каждую модель. Для новой версии iALS меняем только период затухания, остальные параметры берём у лучшего обычного iALS. Так сравнение таргетов будет честным.

In [ ]:
fm_features = MetadataEncoder(data.train, data.user_lookup, data.item_lookup)

params_grid = {
    "ItemKNN": [{"n_neighbors": k, "shrink": 20} for k in [40, 80]],
    "iALS": [{"n_factors": k, "reg": 0.1, "alpha": 15, "n_iters": 6} for k in [24, 32]],
    "EASE": [{"l2": k} for k in [100, 400]],
    "FM": [{"n_factors": k, "learning_rate": 0.008, "epochs": 3,
            "negatives": 2, "batch_size": 8192} for k in [12, 20]]
}

# Одна функция, чтобы не повторять обучение в двух циклах.
def train_model(name, params, counts, quantities, dates, cutoff, features=fm_features):
    if name == "ItemKNN":
        return ItemKNN(**params).fit(counts)
    if name == "iALS":
        return ImplicitALS(seed=SEED, **params).fit(counts)
    if name == "EASE":
        return EASE(**params).fit(counts)
    if name == "FM":
        return FactorizationMachine(features, seed=SEED, **params).fit(counts)
    if name == "CustomTargetIALS":
        return CustomTargetIALS(seed=SEED, **params).fit(counts, quantities, dates, cutoff)
    raise ValueError("Неизвестная модель")

In [ ]:
names = ["ItemKNN", "iALS", "EASE", "FM", "CustomTargetIALS"]
best_params = {}
validation_results = []

for name in names:
    if name == "CustomTargetIALS":
        params_grid[name] = [dict(best_params["iALS"], half_life_days=h) for h in [45, 120]]

    best_score = (-1, -1)
    for params in params_grid[name]:
        model = train_model(name, params, counts_train, qty_train, dates_train,
                            data.train_cut.timestamp())
        metrics = evaluate(model, valid, k=10)
        score = (metrics["Recall@10"], metrics["NDCG@10"])

        validation_results.append({"Модель": name, "Параметры": str(params),
                                   "Recall@10": score[0], "NDCG@10": score[1]})
        print(name, params, "Recall@10 =", round(score[0], 4))
        if score > best_score:
            best_score = score
            best_params[name] = params

display(pd.DataFrame(validation_results))
print("Лучшие параметры:")
display(pd.DataFrame([{"Модель": name, "Параметры": str(best_params[name])} for name in names]))

## 6. Итоговое сравнение на test

Теперь обучаем каждую модель на train + validation с выбранными параметрами. Test до этого не использовали. Релевантность одинаковая для всех моделей: покупка нового товара.

In [ ]:
test_results = []
for name in names:
    model = train_model(name, best_params[name], counts_train_val, qty_train_val,
                        dates_train_val, data.val_cut.timestamp())
    metrics = evaluate(model, test, k=10)
    test_results.append({"Модель": name, "Параметры": str(best_params[name]),
                         "Recall@10": metrics["Recall@10"],
                         "NDCG@10": metrics["NDCG@10"]})

results = pd.DataFrame(test_results).sort_values("Recall@10", ascending=False)
display(results)

results.set_index("Модель")[["Recall@10", "NDCG@10"]].plot.bar(figsize=(9, 4), rot=20)
plt.title("Тестовые метрики, top-10")
plt.tight_layout()
plt.show()

## 7. Проверяем влияние метапризнаков и нового таргета

Для FM отдельно обучим вариант только с ID пользователя и товара (с теми же параметрами). Новый таргет сравниваем с обычным iALS по итоговой таблице.

In [ ]:
id_features = IDOnlyEncoder(n_users, n_items)
fm_id = train_model("FM", best_params["FM"], counts_train_val,
                    qty_train_val, dates_train_val, data.val_cut.timestamp(),
                    features=id_features)
id_scores = evaluate(fm_id, test, k=10)

fm_scores = results.set_index("Модель").loc["FM"]
display(pd.DataFrame([
    {"FM": "Только ID", "Recall@10": id_scores["Recall@10"],
     "NDCG@10": id_scores["NDCG@10"]},
    {"FM": "С метапризнаками", "Recall@10": fm_scores["Recall@10"],
     "NDCG@10": fm_scores["NDCG@10"]}
]))

## 8. Выводы

In [ ]:
scores = results.set_index("Модель")
print("Лучший метод по Recall@10:", results.iloc[0]["Модель"])
print("Recall@10:", round(results.iloc[0]["Recall@10"], 4))

fm_difference = fm_scores["Recall@10"] - id_scores["Recall@10"]
als_difference = scores.loc["CustomTargetIALS", "Recall@10"] - scores.loc["iALS", "Recall@10"]
print("Разница FM с метаданными и FM только с ID:", round(fm_difference, 4))
print("Разница iALS с новым таргетом и обычного iALS:", round(als_difference, 4))

print("Ограничения: оцениваем только пользователей и товары, которые были в train;")
print("в данных есть покупки, но нет просмотров и явных отрицательных оценок;")
print("результат зависит от одного временного разбиения и выбора популярных товаров.")